# DeepCF 结果分析

本 notebook 用于加载训练好的模型并进行可视化分析：
- t-SNE 嵌入可视化
- 影响力排序与 Top-K 商户识别
- 训练曲线分析

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from deepcf.config import DeepCFConfig
from deepcf.data.generator import generate_synthetic_data
from deepcf.data.utils import scale_features
from deepcf.model.vgae import DeepCFVGAE
from deepcf.eval.ranking import compute_radiation_scores, top_k_merchants, rank_all_merchants
from deepcf.eval.visualize import plot_tsne_embeddings, plot_influence_distribution, plot_training_curves

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. 加载模型

In [ ]:
config = DeepCFConfig()
config.data.num_nodes = 300

CHECKPOINT_PATH = "outputs/test_run/best_model.pt"

model = DeepCFVGAE(config.model)
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

print(f"Loaded checkpoint from epoch {ckpt.get('epoch', 'unknown')}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. 生成数据并获取嵌入

In [ ]:
data = generate_synthetic_data(
    num_nodes=config.data.num_nodes,
    num_features=config.model.input_dim,
    seed=config.seed,
)
X = scale_features(data["features"])
A = data["adjacency"]
W = data["weights"]
labels = data["labels"]

edge_list = []
for i in range(config.data.num_nodes):
    for j in range(i + 1, config.data.num_nodes):
        if A[i, j] > 0:
            edge_list.append([i, j])
            edge_list.append([j, i])

edge_index_np = np.array(edge_list).T
x_tensor = torch.tensor(X, dtype=torch.float32).to(device)
edge_index_tensor = torch.tensor(edge_index_np, dtype=torch.long).to(device)

with torch.no_grad():
    z = model.get_embeddings(x_tensor, edge_index_tensor).cpu().numpy()

print(f"Embedding shape: {z.shape}")

## 3. t-SNE 嵌入可视化

左侧按商户类别着色，右侧按辐射力评分着色。红圈标注 Top-K 关键商户。

In [ ]:
rank_scores = rank_all_merchants(model, x_tensor, edge_index_tensor)
radiation = compute_radiation_scores(z, A, W, rank_scores)
top_k = top_k_merchants(radiation, labels, k=20)
topk_idx = [m["id"] for m in top_k[:10]]

plot_tsne_embeddings(z, labels, radiation, topk_idx, save_path="outputs/notebook_tsne.png")
print("t-SNE visualization saved to outputs/notebook_tsne.png")

## 4. 影响力分布与 Top-K 商户

In [ ]:
plot_influence_distribution(radiation, A.sum(axis=1), topk_idx, save_path="outputs/notebook_influence.png")
print("Influence distribution saved to outputs/notebook_influence.png")

In [ ]:
print("=== Top-10 Key Merchants ===")
for m in top_k[:10]:
    print(f"  ID={m['id']:4d}  Score={m['score']:.4f}  Category={m['category']}")

## 5. 影响力评分统计

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(radiation, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(x=np.median(radiation), color="red", linestyle="--", label=f"Median={np.median(radiation):.3f}")
ax.axvline(x=np.mean(radiation), color="orange", linestyle="--", label=f"Mean={np.mean(radiation):.3f}")
ax.set_title("Radiation Score Distribution"); ax.set_xlabel("Score"); ax.set_ylabel("Count")
ax.legend(); plt.show()

print(f"Mean: {radiation.mean():.4f}")
print(f"Std:  {radiation.std():.4f}")
print(f"Max:  {radiation.max():.4f}")
print(f"Min:  {radiation.min():.4f}")

## 6. 训练曲线（从 checkpoint 中加载）

In [ ]:
history = ckpt.get("history", {"epoch": [], "train_loss": [], "lr": []})
if history["epoch"]:
    plot_training_curves(history, save_path="outputs/notebook_training.png")
    print("Training curves saved to outputs/notebook_training.png")
else:
    print("No training history found in checkpoint.")

print("\nAnalysis complete!")